# Multi-microphone, multi-modal emotion recognition on a Mac

The whole pipeline of the paper on an Apple-silicon Mac: download RAVDESS → build the reverberant 3-microphone datasets → fine-tune the extended HTS-AT (audio) + R(2+1)D (video) → average the best checkpoints → test in simulated rooms and, optionally, the real ACE rooms → predict one clip.

It follows HTS-AT's [`htsat_esc_training.ipynb`](https://github.com/RetroCirce/HTS-Audio-Transformer/blob/main/htsat_esc_training.ipynb); the data preparation is `prep_ravdess.ipynb` adapted to this repo, in `prepare_data.py`. Each step is also a terminal command (in the section titles), which is handier for the long training run.

**Setup, once**
```bash
conda create -n mer python=3.12 -y && conda activate mer
pip install -r requirements-mac.txt
python -m ipykernel install --user --name mer   # then select the "mer" kernel
```

**Downloads**, cached under `data/` (re-running skips what is there):

| what | size |
|---|---|
| RAVDESS speech audio, all actors | 208 MB |
| RAVDESS speech video | ~550 MB per actor, 13.4 GB for all 24 (only the audio-video `.mp4`s are kept) |
| HTS-AT AudioSet checkpoint | 121 MB |
| R(2+1)D Kinetics-400 weights (torchvision, automatic) | 120 MB |
| ACE "Mobile" RIRs (optional) | 1.6 GB |

Set `QUICK = True` below to check everything on 3 actors and 1 epoch first.

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # before torch: run any op the Apple GPU lacks on the CPU
import numpy as np
import torch
import config
import prepare_data as prep

QUICK = False      # True: actors 1-3 and one epoch, to check the whole pipeline end to end
BUILD_ACE = False  # True: also download the ACE "Mobile" RIRs (1.6 GB) and build the real-room test sets
ACTORS = [1, 2, 3] if QUICK else list(range(1, 25))
WORKERS = prep.default_workers()

print("torch", torch.__version__, "| Apple GPU (MPS):", torch.backends.mps.is_available())
print("data:", config.data_root, "| outputs:", config.workspace)

## 1. Download — `python prepare_data.py --download --checkpoint`

RAVDESS (Livingstone & Russo, 2018; CC BY-NC-SA 4.0) from Zenodo, and `HTSAT_AudioSet_Saved_1.ckpt`, the AudioSet checkpoint HTS-AT's ESC-50 tutorial fine-tunes from. The paper's runs started from `HTSAT_AudioSet_Saved_3.ckpt` (in the paper's Dropbox): to use it, point `config.resume_checkpoint` at it.

In [ ]:
prep.download_audioset_checkpoint(config.resume_checkpoint)
audio_dir, video_dir = prep.download_ravdess(ACTORS)

## 2. Build the datasets — `python prepare_data.py --rev`

`prep_ravdess.ipynb` (first paper), adapted to this repo:

| `prep_ravdess.ipynb` | here |
|---|---|
| 7 classes (neutral and calm merged) | 8 classes, `target = emotion code - 1` |
| `{"name", "target", "waveform"}` | `{"wav_name", "video_name", "waveform", "target"}`, plus `actor`, `t60`, `room` |
| stratified random 80/10/10 split | actor split 80/10/10, i.e. 19 / 2 / 3 actors, none shared between splits |
| reads wavs reverberated beforehand with gpuRIR (CUDA only) | simulates the rooms here, on the CPU, with pyroomacoustics |

Each split is still an `(N, 1)` object array whose `[i][0]` is the row. Every utterance gets its own room, as in the paper: length and width ~ U(3, 8) m with ratio ≤ 1.6, height 2.9 m, T60 ~ U(0.5, 0.85) s, the source at 1.75 m and ≥ 0.5 m from the walls, 3 microphones at 1.6 m, each U(0.2 m, critical distance) from the source, and AR(1) noise at 20 dB SNR. The parameters are in `prep.SIM`.

In [ ]:
records = prep.list_ravdess(audio_dir, video_dir, actors=ACTORS)
split = prep.actor_split(sorted({r["actor"] for r in records}), seed=0)
print("actor split:", split)
prep.build_dataset(records, split, config.dataset_path, kind="rev", workers=WORKERS)

In [ ]:
train = np.load(os.path.join(config.dataset_path, "ravdess_av_train.npy"), allow_pickle=True)
row = train[0][0]  # same layout as prep_ravdess.ipynb wrote
print(train.shape, {k: getattr(v, "shape", v) for k, v in row.items()})

### Optional: the real ACE rooms — `python prepare_data.py --download-ace`

The test actors convolved with the RIRs of the ACE corpus' "Mobile" phone (3 microphones) in 7 rooms × 2 positions, written to `config.ace_path` for `main.test_ace()`.

In [ ]:
if BUILD_ACE:
    rir_dir = prep.download_ace_rirs()
    test_records = [r for r in records if r["actor"] in split["test"]]
    prep.build_ace_sets(test_records, rir_dir, config.ace_path, workers=WORKERS)

### Video frames — `python prepare_data.py --video-cache`

Every clip's 8 frames, decoded and resized once into `config.video_cache_dir` (1.2 MB per clip). Training then reads them instead of decoding 720p `.mp4`s; the tensors are bit-for-bit the ones it would compute.

In [ ]:
prep.build_video_cache(records, video_dir, workers=WORKERS)

## 3. Fine-tune — `python main.py train`

On the Mac the model runs on the Apple GPU (MPS), in steps of 8 clips with gradients accumulated over 4 steps: the paper's batch of 32 doesn't fit in memory, and 4 × 8 keeps the same effective batch. R(2+1)D's convolutions run as equivalent 2-D convolutions (same weights, 2.6× faster on MPS). Expect about 2 min per epoch on the full dataset (M4 Max); early stopping (patience 12 on the validation loss) ends most runs long before `max_epoch`.

The 8 best checkpoints (validation accuracy) are saved in `results_.../<exp_name>/checkpoint/checkpoints/`, with CSV logs next to them. For W&B, run `wandb login` once and set `config.use_wandb = True`.

In [ ]:
import main

config.use_wandb = False
if QUICK:
    config.max_epoch = 1
main.train()

## 4. Average and test — `python main.py weight_average`, `python main.py test`, `python main.py test_ace`

In [ ]:
main.weight_average()                      # the saved checkpoints -> config.wa_model_path
config.resume_checkpoint_test = config.wa_model_path
main.test()                                # the test actors in simulated rooms
if BUILD_ACE:
    main.test_ace()                        # the test actors in the 14 ACE rooms -> CSV in config.ace_results_dir

## 5. Predict one clip

As in the last cells of `htsat_esc_training.ipynb`: load the averaged model and classify one test clip from its 3 microphone signals and its video.

In [ ]:
from data_generator import Emotion_DataLoader

test_set = np.load(os.path.join(config.dataset_path, "ravdess_av_test.npy"), allow_pickle=True)
dataset = Emotion_DataLoader(test_set, config, eval_mode=True)
model = main.build_model(dataset)
main.load_weights(model, config.wa_model_path)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device).eval()

sample = dataset[0]
with torch.no_grad():
    logits, _ = model(torch.from_numpy(sample["waveform"])[None].to(device),
                      sample["transformed_video"][None].to(device))
probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
print(sample["audio_name"], "| true:", prep.EMOTIONS[sample["target"]],
      "| predicted: %s (p = %.2f)" % (prep.EMOTIONS[probs.argmax()], probs.max()))